In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer
import torch
import pandas as pd
from datasets import Dataset
from transformers import DataCollatorWithPadding
from sklearn.metrics import classification_report

In [3]:
big_path = "Data/big_data.csv"
smaller_path = "Data/smaller_data.csv"

df = pd.read_csv(smaller_path)

df.rename(columns={'label': 'labels'}, inplace=True)

In [4]:
model_id = "answerdotai/ModernBERT-base"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=1024
    )

hf_dataset = Dataset.from_pandas(df)

# 2. Apply the tokenization
# (batched=True is crucial here so it processes chunks of text at once)
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

split_datasets = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 5230.29it/s]
ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 15000/15000 [00:01<00:00, 9765.62 examples/s] 


In [5]:
print(df['labels'].unique())

[1 0]


In [ ]:
#LOADING MODEL, PRETRAINED
from transformers import AutoModelForSequenceClassification, AutoTokenizer

save_path = "./modernBERT-final"

my_model = AutoModelForSequenceClassification.from_pretrained(save_path)
my_tokenizer = AutoTokenizer.from_pretrained(save_path)

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./modernbert-fake-news",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.085408,0.075156,0.980000
2,0.043901,0.086478,0.981000
3,0.008047,0.136907,0.980000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]


TrainOutput(global_step=4500, training_loss=0.05833432330025567, metrics={'train_runtime': 1953.4665, 'train_samples_per_second': 18.429, 'train_steps_per_second': 2.304, 'total_flos': 1.0921624398317472e+16, 'train_loss': 0.05833432330025567, 'epoch': 3.0})

In [8]:
import torch
print(torch.cuda.is_available())

True


In [13]:
test_results = trainer.predict(split_datasets["test"])

predicted_labels = np.argmax(test_results.predictions, axis=-1)
actual_labels = test_results.label_ids

print(classification_report(actual_labels, predicted_labels, target_names=["real", "fake"]))

              precision    recall  f1-score   support

        real       0.99      0.97      0.98      1489
        fake       0.97      0.99      0.98      1511

    accuracy                           0.98      3000
   macro avg       0.98      0.98      0.98      3000
weighted avg       0.98      0.98      0.98      3000



In [ ]:
save_path = "./modernBERT-final"

trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


('./modernBERT-final\\tokenizer_config.json',
 './modernBERT-final\\tokenizer.json')